This notebook runs linear regression (baseline model) using noisy motor units spike trains during inference.

To get the noisy spike trains, I rely on the same backend used with the SNN models.
This explains why I load methods related to SNN.

In [ ]:
import os
import sys
notebook_dir = os.getcwd()
project_dir = os.path.dirname(os.path.dirname(notebook_dir))

if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

import logging
from hydra import initialize_config_dir, compose

from force_regression.utils.configuration import load_project_configuration, load_input_data_configuration
from force_regression.training.snn_pipeline import SnnMusDatasetPreparation
import force_regression.training.snn_pipeline as snn_prep
from force_regression.models.linear_regression import ConventionalRegressionOnMUwithNoise
import force_regression.utils.configuration as lrc
logging.getLogger().setLevel(logging.INFO)
%load_ext autoreload
%autoreload 2

# Load decomposed MUs

In [ ]:
# UPDATE THESE PARAMETERS BEFORE RUNNING THE NOTEBOOK
subject = 'S1'  # S1 | S2
exp_dir = 'flex'    # flex | ext
percent_omission = 0
percent_addition = 0
percent_misattribution = 10
noise_mode = 'misattribution'  # None| 'omission' | 'addition' | 'misattribution'
decoder_type = 'spiking'

load_preprocessed_data = True # if True load pre-generated preprocessed MU and force dataframes from file, else generate it
win_size_in_sec = 0.08
emg_type = 'intra'
post_process= True
load_regression_data_from_file = True # if True, load the regression data (X and y) from file, else generate it by windowing the MU and force data.


# Prepare Configurations

In [ ]:
abs_config_dir=os.path.abspath("../../configs/hydra")
with initialize_config_dir(version_base=None, config_dir=abs_config_dir):
    cfg = compose(config_name="snn",
                  overrides=[f"decoder_type={decoder_type}",
                             f"task.exp_dir={exp_dir}",
                             f"task.percent_misattribution={percent_misattribution}",
                            f"task.percent_addition={percent_addition}",
                            f"task.percent_omission={percent_omission}",
                            f"task.noise_mode={noise_mode}",
                            "logging.wandb.use_wandb=False",
                             ]
                  )
    print(cfg)
flat_snn_config = lrc.flatten_hydra_config(cfg)


In [ ]:
config_path = os.path.join(project_dir, 'configs/config.json')
data_root_dir, results_root_dir, subject_name_encoding, wandb_entity = load_project_configuration(config_path)

data_config = load_input_data_configuration(flat_snn_config, data_root_dir,
                                                results_root_dir, flat_snn_config.segment_hold,
                                                subject_name_encoding)
output_figures_dir = os.path.join(data_config.root_results_dir, data_config.figs_dir)
mvcs_string = snn_prep.create_mvcs_string(data_config)
snn_config = snn_prep.create_wandb_config(flat_snn_config, wandb_entity) if flat_snn_config.use_wandb else flat_snn_config
print(f"Config from file: {snn_config}")

print(data_config.results_path, output_figures_dir)


# Load prepared MUs df

In [ ]:
mu_df_sorted, force_df, data_config = snn_prep.load_mu_data_for_snn(data_config,
                                                                    load_preprocessed_data,
                                                                    mvcs_string,
                                                                    flat_snn_config)
prepared_snndata = SnnMusDatasetPreparation(mu_df_sorted.copy(),
                                            force_df,
                                            data_config,
                                            flat_snn_config,
                                            flat_snn_config.select_dir,
                                            'train',
                                            flat_snn_config.shuffle_fingers_seed)
offline_dataset = prepared_snndata.create_snn_dataset()

# Fit model

In [ ]:
model_type ='linear'
if noise_mode == 'omission':
    percent_noise = percent_omission
elif noise_mode == 'addition':
    percent_noise = percent_addition
elif noise_mode == 'misattribution':
    percent_noise = percent_misattribution
ConvRegNoise = ConventionalRegressionOnMUwithNoise(linear_model=model_type,
                                    data_config=data_config,
                                    emg_type=emg_type,
                                    snndata = prepared_snndata,
                                    snn_dt=flat_snn_config.dt,
                                    regression_data_parent_dir=data_config.convreg_temp_data_path,
                                    overlap_in_perc=flat_snn_config.overlap_perc * 100,
                                    window_size_in_sec=win_size_in_sec,
                                    post_process=post_process,
                                    load_regression_data_from_file=load_regression_data_from_file,
                                    shuffle_fingers_seed=flat_snn_config.shuffle_fingers_seed,
                                    percent_noise = percent_noise,
                                    noise_mode= noise_mode,
                                    )

In [ ]:
ConvRegNoise.prepare_regression_dataframes()


In [ ]:
metrics_df_cv, y_df_cv, trained_models_df = ConvRegNoise.cross_validate_model()

In [ ]:
postprocessed_y_dict = ConvRegNoise.post_process_y_cv_df(y_df_cv)
postprocessed_y_dict

In [ ]:
print(f"Saving results to: {ConvRegNoise.data_config.results_path}")
ConvRegNoise.save_results(metrics_df_cv, y_df_cv, postprocessed_y_dict, trained_models_df,
                          is_noise_experiment=True,
                          noise_mode=noise_mode
                          )